In [1]:
import os
import getpass

from langchain_openai import ChatOpenAI

GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or getpass.getpass("Groq API key: ")

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

In [ ]:
from pathlib import Path
from pypdf import PdfReader
from docx import Document
import pandas as pd

FOLDER_PATH = Path("material")
SUPPORTED = {".pdf", ".txt", ".docx", ".csv"}
MAX_CHARS = 15000 


def read_file(file_path):
    extension = file_path.suffix.lower()

    if extension == ".pdf":
        reader = PdfReader(file_path)
        return "\n".join(page.extract_text() or "" for page in reader.pages)

    elif extension == ".txt":
        return file_path.read_text(encoding="utf-8")

    elif extension == ".docx":
        document = Document(file_path)
        return "\n".join(paragraph.text for paragraph in document.paragraphs)

    elif extension == ".csv":
        return pd.read_csv(file_path).to_string(index=False)

    else:
        raise ValueError("File type not supported")


def get_files():
    if not FOLDER_PATH.exists():
        return []
    return sorted(
        f for f in FOLDER_PATH.iterdir()
        if f.is_file() and f.suffix.lower() in SUPPORTED
    )

In [ ]:
import json
import re
from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool

quiz = {}


def reset_quiz():
    quiz.clear()
    quiz.update({
        "document": "",
        "topic": "",
        "questions": [],
        "current_question": 0,
        "score": 0,
        "correct_answers": 0,
        "wrong_answers": 0,
        "weak_topics": [],
        "finished": False,
    })


reset_quiz()


def format_question(index):
    q = quiz["questions"][index]
    options = "\n".join(f"{key}) {value}" for key, value in q["options"].items())
    return f"Question {index + 1}/{len(quiz['questions'])}: {q['question']}\n{options}"


def parse_questions(raw):
    raw = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()
    questions = json.loads(raw)["questions"]
    for q in questions:
        assert set(q["options"].keys()) == {"A", "B", "C", "D"}, "options must be A-D"
        assert q["correct_answer"].strip().upper() in {"A", "B", "C", "D"}
        q["correct_answer"] = q["correct_answer"].strip().upper()
        for field in ("topic", "question", "explanation"):
            assert q[field]
    return questions


QUIZ_PROMPT = ChatPromptTemplate.from_template("""
You are an educational assessment designer.

Create questions using the document below.

Topic: {topic}

Document:
{document}

Requirements:
- Each question must have exactly 4 options: A, B, C, D.
- Only one option can be correct.
- Cover important concepts from the document.
- Include the correct answer and a clear explanation.
- Do not invent facts that are unsupported by the document.
- Make questions test understanding, not just memorization.

Return ONLY valid JSON (no markdown) with this structure:
{{
    "questions": [
        {{
            "topic": "concept name",
            "question": "question text",
            "options": {{"A": "option A", "B": "option B", "C": "option C", "D": "option D"}},
            "correct_answer": "A",
            "explanation": "why this answer is correct"
        }}
    ]
}}
""")


@tool
def list_files() -> str:
    """List the study files available in the material folder, numbered."""
    files = get_files()
    if not files:
        return "No supported files found in the material folder."
    return "\n".join(f"{i}. {f.name}" for i, f in enumerate(files, start=1))


@tool
def load_file(file_number: int) -> str:
    """Load the file with the given number (from list_files) and extract its text."""
    files = get_files()
    if not 1 <= file_number <= len(files):
        return "Error: Invalid file number."
    selected = files[file_number - 1]
    try:
        text = read_file(selected)
    except Exception as error:
        return f"Error: could not read file ({error})"
    if not text.strip():
        return "Error: No text extracted. The PDF might be scanned."
    reset_quiz()
    quiz["document"] = text
    return f"Selected: {selected.name}. Extracted {len(text)} characters."


@tool
def generate_quiz(topic: str) -> str:
    """Generate multiple-choice questions from the loaded file about the given topic,
    and return the first question."""
    if not quiz["document"]:
        return "Error: no file loaded yet. Ask the user to choose a file first."
    chain = QUIZ_PROMPT | llm
    last_error = None
    for _ in range(3):
        try:
            response = chain.invoke(
                {"document": quiz["document"][:MAX_CHARS], "topic": topic}
            )
            questions = parse_questions(response.content)
            break
        except Exception as error:
            last_error = error
    else:
        return f"Error: could not generate the quiz ({last_error}). Try again."

    document = quiz["document"]
    reset_quiz()
    quiz.update({"document": document, "topic": topic, "questions": questions})
    return "Quiz ready.\n\n" + format_question(0)


@tool
def submit_answer(answer: str) -> str:
    """Check the user's answer (A, B, C or D) for the current question, then return
    the feedback together with the next question, or the final result at the end."""
    if not quiz["questions"]:
        return "Error: no quiz yet. Generate a quiz first."
    if quiz["finished"]:
        return "The quiz is already finished."

    index = quiz["current_question"]
    question = quiz["questions"][index]
    user_answer = answer.strip().upper()
    correct_answer = question["correct_answer"]

    if user_answer not in {"A", "B", "C", "D"}:
        return "Invalid answer. Please choose A, B, C, or D.\n\n" + format_question(index)

    if user_answer == correct_answer:
        quiz["score"] += 1
        quiz["correct_answers"] += 1
        feedback = f"Correct! {question['explanation']}"
    else:
        quiz["wrong_answers"] += 1
        quiz["weak_topics"].append(question["topic"])
        feedback = (
            f"Incorrect. The correct answer is {correct_answer}: "
            f"{question['options'][correct_answer]}. {question['explanation']}"
        )

    # next question or final assessment
    if index + 1 >= len(quiz["questions"]):
        quiz["finished"] = True
        total = len(quiz["questions"])
        percentage = round(quiz["score"] / total * 100)
        weak = ", ".join(sorted(set(quiz["weak_topics"]))) or "None"
        return (
            f"{feedback}\n\nQuiz completed. Score: {quiz['score']}/{total} "
            f"({percentage}%). Review these topics: {weak}"
        )

    quiz["current_question"] = index + 1
    return f"{feedback}\n\n{format_question(index + 1)}"


tools = [list_files, load_file, generate_quiz, submit_answer]

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

SYSTEM_PROMPT = """You are a quiz tutor. Follow these steps in order:
1. Call list_files and show the numbered list to the user, then ask which file number they want.
2. When they answer, call load_file with that number.
3. Ask the user for the quiz topic (if they did not give one), then call generate_quiz.
4. Show the question exactly as the tool returned it and wait for the user's answer.
5. Each time the user answers, call submit_answer with their answer and show the tool result exactly as returned.
6. When the tool says the quiz is completed, show the final result and stop.
Never invent questions, answers or scores yourself. Always rely on the tools."""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)

In [5]:
def reply_text(result):
    content = result["messages"][-1].content
    if isinstance(content, list):
        return "".join(b.get("text", "") for b in content if isinstance(b, dict))
    return content


config = {"configurable": {"thread_id": "student-001"}}

reset_quiz()
user_input = "Hi, I want to start a quiz."

while True:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config,
    )
    print("\nAgent:", reply_text(result))

    if quiz["finished"]:
        break

    user_input = input("\nYou: ")
    if user_input.strip().lower() in {"exit", "quit"}:
        break


Agent: Here are the available files:

1. Automata 1.pdf.pdf  
2. Automata 2.pdf.pdf  
3. Trend AI 1.pdf.pdf  
4. Trend AI 2.pdf.pdf  

Which file number would you like to use for the quiz?

Agent: Great! Please let me know the quiz topic you'd like to cover (e.g., “Finite Automata,” “Regular Expressions,” or any specific concept from Automata 2). If you’re not sure, I can pick a common topic for you.

Agent: **Question 1/4:**  
Which of the following strings would NOT be accepted by the DFA described in the document for strings containing the substring 01?  

A) 0011  
B) 0110  
C) 1100  
D) 0101  

Please reply with your answer choice (A, B, C, or D).

Agent: **Question 2/4:**  
Which of the following strings is NOT in the language L = {a w a : w ∈ {a,b}*}?  

A) aaa  
B) aba  
C) baa  
D) aab  

Please reply with your answer choice (A, B, C, or D).

Agent: **Question 3/4:**  
Which of the following strings would NOT be accepted by the DFA that recognizes strings containing at least 